# Day 2.2 Airflow: Run the Day 1 API Ingestion as Pipelines

In Day 1, you collected real API data with Python scripts and notebooks.

In this notebook, you will run similar work with Apache Airflow:

```text
API request -> save raw JSON -> clean data -> load DuckDB -> inspect output
```

The difference is that Airflow gives the workflow a visible structure:

```text
DAG -> tasks -> dependencies -> logs -> retries -> schedule
```

Day 1 now has three real sources: taxi availability, 2-hour weather forecast, and rainfall. This Airflow exercise starts with taxi and weather so the orchestration concepts stay clear. Rainfall can be added later with the same pattern.


## 0. Review the One-Image Dashboard

Before starting Airflow, review the simplest Day 1 dashboard once. In the Day 1 folder, this is the optional live-serving dashboard under `optional_live_serving/`.

This dashboard is enough for today's lesson because it has one clear output: a map image. When the page refreshes, the backend collects fresh API data and redraws the image.

```text
API sources -> Python collection code -> latest map image -> browser
```

Airflow will not replace the dashboard. Airflow will manage the data collection workflow. The dashboard gives us a simple place to see whether the collected data is changing.

Open a terminal in the course folder and activate `.venv` first.

Windows PowerShell:

```powershell
.\.venv\Scripts\Activate.ps1
cd day_1\04_serving_dashboard\optional_live_serving
```

macOS Terminal:

```bash
source ./.venv/bin/activate
cd day_1/04_serving_dashboard/optional_live_serving
```

After activation, run the dashboard server. We use port `8051` here so it does not collide with the Day 1 example port `8010` if that server is still open.

```bash
python -m uvicorn simple_live_serving_app:app --reload --host 127.0.0.1 --port 8051
```

Open:

```text
http://127.0.0.1:8051
```

What to notice:

- The page mainly shows one map image.
- The image comes from `/map.png`.
- The browser refreshes the data every few minutes.
- Each refresh runs backend Python code, then redraws the image.

That is all we need here. Later, Airflow will manage the collection process, and this kind of dashboard will be the place where users see the result.

Stop the server later with `Ctrl + C` in the terminal.

### If the port is already in use

If `uvicorn` says the port is already in use, it is usually an old example server that is still running.

First check which process is using the port.

Windows PowerShell:

```powershell
netstat -ano | Select-String ":8051"
```

macOS Terminal:

```bash
lsof -i :8051
```

If you recognize it as an old course example process, stop it.

Windows PowerShell, replace `PID` with the number from the last column:

```powershell
taskkill /PID PID /F
```

macOS Terminal, replace `PID` with the process ID:

```bash
kill -9 PID
```

Then run the same `python -m uvicorn ...` command again.


## 0. Where the Dashboard Refresh Happens

The dashboard page itself is simple. The important update logic is in `day_1/04_serving_dashboard/optional_live_serving/simple_live_serving_app.py`.

When the browser refreshes the dashboard:

```text
browser -> /api/refresh -> refresh() -> collect_once() -> fetch APIs -> save data -> redraw /map.png
```

For today's Airflow lesson, this is the key design point:

- Before Airflow: the dashboard backend collects API data by itself.
- After Airflow: Airflow should manage data collection, and the dashboard should mainly read the latest prepared output.

So if we connect this dashboard to Airflow output later, the main place to inspect is `collect_once()` and the endpoints that call it.


In [2]:
from pathlib import Path
import ast

candidates = [
    Path("../../day_1/04_serving_dashboard/optional_live_serving/simple_live_serving_app.py"),
    Path("day_1/04_serving_dashboard/optional_live_serving/simple_live_serving_app.py"),
]
app_path = next(path for path in candidates if path.exists())
source = app_path.read_text(encoding="utf-8")
module = ast.parse(source)

for name in ["map_png", "refresh", "collect_once"]:
    node = next(n for n in module.body if isinstance(n, ast.FunctionDef) and n.name == name)
    lines = source.splitlines()[node.lineno - 1: node.end_lineno]
    print("\n" + "=" * 80)
    print(f"{app_path} :: {name}()")
    print("=" * 80)
    print("\n".join(lines))



..\..\day_1\04_serving_dashboard\optional_live_serving\simple_live_serving_app.py :: map_png()
def map_png() -> FileResponse:
    if not LATEST_MAP_PATH.exists():
        collect_once()
    return FileResponse(LATEST_MAP_PATH, media_type="image/png")

..\..\day_1\04_serving_dashboard\optional_live_serving\simple_live_serving_app.py :: refresh()
def refresh() -> dict[str, Any]:
    try:
        collect_once()
        return map_payload()
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc)) from exc

..\..\day_1\04_serving_dashboard\optional_live_serving\simple_live_serving_app.py :: collect_once()
def collect_once() -> None:
    init_database()
    refresh_started_at = now_sgt()
    try:
        taxi_payload = fetch_json(TAXI_API_URL)
        rainfall_payload = fetch_json(RAINFALL_API_URL)

        taxi_timestamp, taxi_counts, taxi_points = extract_taxi_counts(taxi_payload)
        rainfall_timestamp, rainfall_by_area = extract_rainfall_by_area(rai

## 1. Airflow Orchestrates the Pipeline

The dashboard showed one important problem: data collection is hidden inside a web app refresh.

Airflow gives that collection work a proper workflow:

```text
fetch data -> clean data -> load data -> check result
```

In this lesson, learn Airflow in this order:

1. Check and start the tool.
2. Observe the Airflow services that are running.
3. Open the Airflow UI and observe the DAG.
4. Look at the project files that define the pipeline.
5. Connect the files back to core Airflow concepts.
6. Trigger runs, read logs, inspect outputs, and discuss reliability.

Do not worry if the Airflow terms feel abstract at first. Start the tool first; the concepts will make more sense after you see the UI.


## 2. Install and Check Tools

Before running Airflow, check two things:

1. Docker Desktop is installed and running.
2. The course Python environment `.venv` can be activated.

Check Docker:

```bash
docker --version
docker compose version
```

If either command fails, start Docker Desktop and try again.

Activate the course Python environment from the course folder.

Windows PowerShell:

```powershell
.\.venv\Scripts\Activate.ps1
where python
```

macOS Terminal:

```bash
source ./.venv/bin/activate
which python
```

The Python path should point to the course `.venv`. Use this activated environment for local FastAPI dashboards. Docker runs Airflow in containers, but the dashboard servers on ports `8051` and `8052` run on your local computer.


## 3. Open a Terminal in the Airflow Project

Use one terminal for Docker commands.

Windows PowerShell:

```powershell
cd "day_2\day_2_2_Airflow\airflow_project"
```

macOS Terminal:

```bash
cd day_2/day_2_2_Airflow/airflow_project
```

The exact path depends on where your course folder is stored. The terminal should be inside `airflow_project`, the folder that contains `docker-compose.yaml`.


## 4. Start Airflow

Initialize Airflow metadata first:

```bash
docker compose up airflow-init
```

Here `airflow-init` is the service name defined in `docker-compose.yaml`. This command tells Docker Compose to run only that initialization service, which prepares Airflow's metadata database before the normal Airflow services start.

This command may keep showing logs in the terminal instead of returning to a prompt immediately. Wait until you see that `airflow-init` completed successfully, usually with a message like `exited with code 0`. Then press `Ctrl + C` to get your prompt back, or open a second terminal in the same `airflow_project` folder for the next command.

Start the Airflow services in the background:

```bash
docker compose up -d
```

Check containers:

```bash
docker compose ps
```

Open the Airflow UI:

```text
http://localhost:8080
```

Default login:

```text
Username: airflow
Password: airflow
```

First startup can take a few minutes. If the page does not open immediately, wait and refresh.


## 5. Observe the Airflow Services

After `docker compose up -d`, check which containers are running:

```bash
docker compose ps
```

You should see several services. These names help explain the Airflow architecture:

| Service | What it does |
|---|---|
| `airflow-apiserver` | Hosts the Airflow web UI and API. You open this in the browser. |
| `airflow-scheduler` | Decides when DAG runs should happen and when tasks are ready to start. |
| `airflow-worker` | Executes task code, such as the Python functions in `jobs/`. |
| `postgres` | Stores Airflow metadata: DAG runs, task status, users, and history. |
| `redis` | Helps queue work between Airflow components. |

You do not need to operate these services one by one. For now, just connect the words to something visible:

```text
scheduler decides -> worker executes -> metadata database records what happened
```

Where to observe them:

| Concept | Where you can see it |
|---|---|
| Scheduler | `docker compose ps`, scheduler logs, and scheduled DAG runs appearing in the UI |
| Worker | Task logs in the UI and the `airflow-worker` container |
| Metadata database | Run history, task status, and logs metadata shown in the UI |

### View Scheduler Logs

The scheduler decides when a DAG run should be created and when tasks are ready to run.

From the terminal inside `airflow_project`, run:

```bash
docker compose logs airflow-scheduler --tail=50
```

Look for messages about DAG parsing, scheduling, or queued tasks.

### View Worker and Task Logs

The worker executes the Python task code.

To see the worker container log:

```bash
docker compose logs airflow-worker --tail=50
```

The more useful view is usually inside the Airflow UI:

1. Open the DAG run.
2. Click one task box, such as `join_taxi_to_planning_area`.
3. Open **Logs**.

That task log shows the Python function output and error messages for that specific task attempt.

### View the Metadata Database

Airflow stores DAG runs, task status, users, and history in the metadata database. In this Docker setup, the metadata database is the `postgres` service.

Most of the time, you view metadata through the Airflow UI: Grid view, Graph view, task status, and run history all come from this database.

For a quick read-only look from the terminal, run:

```bash
docker compose exec postgres psql -U airflow -d airflow -c "SELECT dag_id, run_id, state FROM dag_run ORDER BY start_date DESC LIMIT 5;"
```

You can also inspect recent task states:

```bash
docker compose exec postgres psql -U airflow -d airflow -c "SELECT dag_id, task_id, state FROM task_instance ORDER BY start_date DESC LIMIT 10;"
```

Do not update or delete rows in the metadata database during this lesson. Use the Airflow UI or CLI to manage DAG runs.


## 6. Project Files

This folder now has a clean structure. The notebook is the lesson entry point, and `airflow_project/` is the runnable Airflow project:

```text
day_2_2_Airflow/
+-- day_2_2_Airflow.ipynb
+-- airflow_project/
    +-- docker-compose.yaml
    +-- .env
    +-- dags/
    +-- jobs/
    +-- data/
    +-- dashboard/
    +-- logs/
    +-- plugins/
    +-- serving_api.py
```

Important folders:

| Folder | Purpose |
|---|---|
| `airflow_project/dags/` | Airflow scans this folder for workflow definitions. |
| `airflow_project/jobs/` | Python functions called by DAG tasks. |
| `airflow_project/data/` | Raw JSON, cleaned CSV, curated CSV, and DuckDB output. |
| `airflow_project/dashboard/` | Dashboard files and the planning-area GeoJSON used by the join task. |
| `airflow_project/logs/` | Airflow task logs. |


In [3]:
from pathlib import Path

PROJECT_DIR = Path("airflow_project")
expected = [
    "docker-compose.yaml",
    ".env",
    "dags/dashboard_pipeline_dag.py",
    "jobs/taxi_job.py",
    "jobs/weather_job.py",
    "jobs/geo_utils.py",
    "jobs/config.py",
    "dashboard/shared_assets/MasterPlan2019SubzoneBoundaryNoSeaGEOJSON.geojson",
    "jobs/shared_assets/MasterPlan2019SubzoneBoundaryNoSeaGEOJSON.geojson",
]

print("Notebook folder:", Path.cwd())
print("Airflow project folder:", PROJECT_DIR.resolve())
for item in expected:
    path = PROJECT_DIR / item
    print(f"{item:70} {'OK' if path.exists() else 'MISSING'}")


Notebook folder: d:\teaching\slides\day_2\day_2_2_Airflow
Airflow project folder: D:\teaching\slides\day_2\day_2_2_Airflow\airflow_project
docker-compose.yaml                                                    OK
.env                                                                   OK
dags/dashboard_pipeline_dag.py                                         OK
jobs/taxi_job.py                                                       OK
jobs/weather_job.py                                                    OK
jobs/geo_utils.py                                                      OK
jobs/config.py                                                         OK
dashboard/shared_assets/MasterPlan2019SubzoneBoundaryNoSeaGEOJSON.geojson OK
jobs/shared_assets/MasterPlan2019SubzoneBoundaryNoSeaGEOJSON.geojson   OK


## 7. After Changing Files

Airflow reads files from this project folder through Docker volume mounts. If you change files while Airflow is running, what you need to do depends on what changed.

| What changed? | What to do |
|---|---|
| `jobs/*.py` Python logic | Trigger a new DAG run. If the old code still appears, restart Airflow services. |
| `dags/*.py` DAG structure | Wait a short moment and refresh the Airflow UI. If the DAG does not update, restart Airflow services. |
| `.env` | Restart containers so the new environment variables are loaded. |
| `docker-compose.yaml` | Restart containers so Docker uses the new configuration. |
| New Python package requirement | Restart containers. Airflow installs `_PIP_ADDITIONAL_REQUIREMENTS` when containers start. |
| New mounted folder or volume | Recreate containers with `docker compose down` and `docker compose up -d`. |

For normal code or DAG edits, this is usually enough:

```bash
docker compose restart airflow-scheduler airflow-dag-processor airflow-worker
```

For `.env`, `docker-compose.yaml`, package changes, or new volume mounts, use the safer full restart:

```bash
docker compose down
docker compose up -d
```

If this is the first time starting Airflow, or the metadata database was reset, run the init service first. `airflow-init` is a service name from `docker-compose.yaml`; it prepares the Airflow metadata database. If the terminal stays attached to logs after init succeeds, press `Ctrl + C` or use another terminal before running `docker compose up -d`.

```bash
docker compose up airflow-init
docker compose up -d
```

After restarting, refresh the Airflow UI and trigger a new DAG run.


## 8. DAG, Task, and Operator

Now look back at the DAG code and the Airflow UI.

These three concepts are the most important when reading a DAG file:

| Concept | Meaning in this project |
|---|---|
| DAG | The workflow definition. We use `dashboard_data_pipeline`. |
| Task | One visible step in the DAG graph, such as `fetch_taxi_api` or `join_taxi_to_planning_area`. |
| Operator | The type of task. We use `PythonOperator` to run normal Python functions. |

A useful way to read the DAG:

```text
DAG = the whole pipeline
Task = one box in the graph
Operator = how that box runs
```

The other Airflow services, such as scheduler, worker, and metadata database, are runtime components. You observed them with `docker compose ps` before reading the DAG file.


## 9. Create the DuckDB Pool

In real data pipelines, running many tasks at the same time can sometimes cause problems.

For example:

- Multiple tasks may write to the same database file at the same time.
- Too many tasks may call the same API and hit rate limits.
- Too many tasks may connect to the same database and exceed the connection limit.

In this demo, both load tasks may write to the same DuckDB file:

```text
load_taxi_counts_to_duckdb
load_current_weather_to_duckdb
```

DuckDB is a local file-based database. To keep this classroom demo simple and safe, create an Airflow pool with only 1 slot.

This means only one task using this pool can run at a time.

### Create the pool from the Airflow UI

1. Open the Airflow Web UI.
2. Go to **Admin -> Pools**.
3. Click **Add Pool**.
4. Enter these values:

| Field | Value |
|---|---|
| Pool | `data_ingestion_pool` |
| Slots | `1` |
| Description | `Limit ingestion tasks` |

5. Click **Save**.

After the pool is created, tasks can use it by adding the `pool` parameter in the DAG file:

```python
load_weather = PythonOperator(
    task_id="load_current_weather_to_duckdb",
    python_callable=load_weather_to_duckdb,
    op_args=[clean_weather.output],
    pool="data_ingestion_pool",
)
```

This means the task must get one slot from `data_ingestion_pool` before it can run.

Since the pool has only one slot, if another task using the same pool is already running, this task will wait in the queue.

### Optional: Create the Pool from the Command Line

You can also create the same pool using the Airflow CLI:

```bash
docker compose exec airflow-apiserver airflow pools set data_ingestion_pool 1 "Limit ingestion tasks"
```

This creates:

```text
data_ingestion_pool, slots = 1
```

The key idea is:

```text
Pool = a limited number of execution slots for tasks using the same shared resource
```


## 10. Read the DAG File

Open this file:

```text
airflow_project/dags/dashboard_pipeline_dag.py
```

Look for these pieces:

| Code | Meaning |
|---|---|
| `dag_id="dashboard_data_pipeline"` | Name shown in the Airflow UI |
| `schedule` | How often the DAG should run |
| `catchup=False` | Do not backfill old scheduled runs |
| `PythonOperator` | Run a Python function as a task |
| `fetch_taxi >> clean_taxi >> join_taxi_area >> load_taxi` | Taxi task dependencies |
| `fetch_weather >> clean_weather >> load_weather` | Weather task dependencies |
| `pool="data_ingestion_pool"` | Use the one-slot DuckDB pool for load tasks |
| `retries` and `retry_delay` | Try again after temporary failure |

### Backfill and `catchup=False`

**Backfill** means running a pipeline for past time periods.

Example:

```text
Today is June 9.
The pipeline failed on June 7 and June 8.
You run the DAG again for June 7 and June 8 to fill the missing data.
```

In production, backfill is important because data pipelines can fail, upstream systems can be delayed, and historical data may need to be recomputed.

But backfill also requires careful design:

- The task should know which date or time window it is processing.
- Output data should usually be partitioned by date or timestamp.
- Tasks should be idempotent, so rerunning the same period does not create duplicates.
- The source API must support requesting historical data, or you must already have raw data stored.

In this classroom DAG, we use:

```python
catchup=False
```

This keeps the demo simple. Airflow will not automatically create many old scheduled runs from the `start_date`.

For this live API example, `catchup=False` is also realistic: the taxi and current weather endpoints mainly represent the latest state, not a clean historical partition that we can safely replay.


In [ ]:
from pathlib import Path

PROJECT_DIR = Path("airflow_project")
for file_name in ["dags/dashboard_pipeline_dag.py"]:
    path = PROJECT_DIR / file_name
    print("\n" + "=" * 80)
    print(path)
    print("=" * 80)
    print(path.read_text(encoding="utf-8"))


## 11. Read the Job Functions

The DAG file defines orchestration. The `airflow_project/jobs/` files contain the actual data work.

Open:

```text
airflow_project/jobs/taxi_job.py
airflow_project/jobs/weather_job.py
airflow_project/jobs/geo_utils.py
```

The taxi branch now has visible steps:

```text
fetch raw taxi JSON -> clean taxi points -> join coordinates to planning area -> load taxi counts
```

The weather branch is shorter:

```text
fetch current weather JSON -> clean forecasts -> load current weather
```

This is the main Airflow idea: Python still does the work, but Airflow makes the steps, dependencies, retries, and logs visible.


## 12. Find the DAG in the UI

In the Airflow UI:

1. Open `http://localhost:8080`.
2. Search for `dashboard_data_pipeline`.
3. Open the DAG page.
4. Look at the Graph view.
5. Notice the two branches: taxi and weather.

The taxi branch has more tasks because it includes the planning-area join. This is where Airflow starts to look like orchestration instead of just running one script.

New DAGs may be paused. If the DAG is paused, unpause it before triggering.


## 13. Trigger the DAG

First trigger the DAG from the Airflow UI. This is the easiest way to see orchestration.

### Trigger from the Airflow UI

1. Open `http://localhost:8080`.
2. Search for `dashboard_data_pipeline`.
3. If the DAG is paused, switch it to active.
4. Open the DAG page.
5. Click the trigger button.
6. Open **Graph** view or **Grid** view.
7. Watch the task states change.

You should see the taxi branch run in order:

```text
fetch_taxi_api -> clean_taxi_points -> join_taxi_to_planning_area -> load_taxi_counts_to_duckdb
```

You should also see the weather branch run in order:

```text
fetch_current_weather_api -> clean_current_weather -> load_current_weather_to_duckdb
```

If a task fails, click the task and open **Logs**.

### Optional: Trigger from the Command Line

You can also trigger from the terminal.

Unpause:

```bash
docker compose exec airflow-apiserver airflow dags unpause dashboard_data_pipeline
```

Trigger:

```bash
docker compose exec airflow-apiserver airflow dags trigger dashboard_data_pipeline
```

Check recent runs:

```bash
docker compose exec airflow-apiserver airflow dags list-runs dashboard_data_pipeline
```


## 14. Inspect Output Files

After the DAG succeeds, the `airflow_project/data/` folder should contain outputs like:

```text
data/
+-- raw/
|   +-- taxi_raw_....json
|   +-- weather_raw_....json
+-- clean/
|   +-- taxi_clean_....csv
|   +-- weather_clean_....csv
+-- curated/
|   +-- taxi_points_with_area.csv
|   +-- taxi_counts.csv
+-- basic_ingestion.duckdb
```

The exact raw and clean filenames include timestamps. The curated files are stable names because the dashboard reads the latest prepared output.


In [ ]:
from pathlib import Path

PROJECT_DIR = Path("airflow_project")
folders = [
    PROJECT_DIR / "data/raw",
    PROJECT_DIR / "data/clean",
    PROJECT_DIR / "data/curated",
]

for folder in folders:
    print()
    print(folder)
    if folder.exists():
        for path in sorted(folder.glob("*"))[-10:]:
            print(" -", path.name)
    else:
        print("Folder does not exist yet.")

print()
print("DuckDB exists:", (PROJECT_DIR / "data/basic_ingestion.duckdb").exists())


## 15. Inspect DuckDB Output

The database file is:

```text
airflow_project/data/basic_ingestion.duckdb
```

Run the next cell after the DAG has succeeded.


In [ ]:
from pathlib import Path

import duckdb

PROJECT_DIR = Path("airflow_project")
DB_PATH = PROJECT_DIR / "data/basic_ingestion.duckdb"

if not DB_PATH.exists():
    print("DuckDB file not found yet. Run the DAG first.")
else:
    with duckdb.connect(str(DB_PATH), read_only=True) as conn:
        print("Tables:")
        print(conn.execute("SHOW TABLES").fetchall())

        print("\nTaxi counts sample:")
        print(conn.execute("SELECT * FROM taxi_area_counts LIMIT 5").df())

        print("\nWeather sample:")
        print(conn.execute("SELECT * FROM weather_forecasts LIMIT 5").df())


## 16. Start the Airflow Output Dashboard

After the Airflow DAG writes `airflow_project/data/basic_ingestion.duckdb`, start the small serving API from the project folder.

This dashboard is not the Airflow UI. It is a user-facing view of the data produced by Airflow.

Keep Airflow running. Open a second terminal in the course folder and activate `.venv` first.

Windows PowerShell:

```powershell
.\.venv\Scripts\Activate.ps1
cd day_2\day_2_2_Airflow\airflow_project
```

macOS Terminal:

```bash
source ./.venv/bin/activate
cd day_2/day_2_2_Airflow/airflow_project
```

After activation, run:

```bash
python -m uvicorn serving_api:app --reload --host 127.0.0.1 --port 8052
```

Open:

```text
http://127.0.0.1:8052
http://127.0.0.1:8052/docs
```

You now have two browser pages with different purposes:

| URL | Purpose |
|---|---|
| `http://localhost:8080` | Airflow UI: pipeline runs, task status, logs |
| `http://127.0.0.1:8052` | Dashboard: data output produced by the pipeline |

The serving API is already configured to read:

```text
airflow_project/data/basic_ingestion.duckdb
```

If the dashboard says the DuckDB file is missing, run the Airflow DAG first and wait until it succeeds.


## 17. What Changed From Day 1?

| Day 1 | Airflow version |
|---|---|
| You run code manually | Airflow triggers DAG runs |
| Code order is inside a notebook/script | `fetch -> clean -> join -> load` is visible in the DAG graph |
| Errors may be buried in terminal output | Each task has logs |
| Rerun usually means rerun the whole script | You can retry or clear one task |
| Timing is manual or a simple loop | Schedule is part of the DAG |
| Data files are local outputs | Raw, clean, curated, and DuckDB outputs are task outputs in `airflow_project/data/` |

Key idea:

```text
Airflow does not replace your data logic.
Airflow makes the data logic scheduled, observable, and recoverable.
```


## 18. Idempotency: Can We Safely Rerun a Failed Task?

A common pipeline question is:

```text
A task failed halfway. Should we run it again?
```

In Airflow, the answer should usually be yes. Airflow is valuable because it can retry a failed task, or let you clear and rerun one failed task from the UI.

But this only works safely if the task is designed to be **idempotent**.

**Idempotency** means rerunning the same task produces the same correct final result.

This is mainly a **task-level** requirement:

- If `fetch_taxi_api` runs twice, it should not corrupt raw data.
- If `join_taxi_to_planning_area` runs twice, it should produce the same curated output for the same input.
- If `load_taxi_counts_to_duckdb` runs twice, it should not duplicate the same rows.

It also affects the whole DAG. Safe tasks make retries, manual reruns, and backfills much easier.

### Failure Scenario

Imagine this run:

```text
fetch_taxi_api: success
clean_taxi_points: success
join_taxi_to_planning_area: success
load_taxi_counts_to_duckdb: failed
```

With Airflow, you do not need to rerun everything manually. You can inspect the failed task log, fix the issue, and rerun the failed task or rerun the DAG.

The risk is duplicate output.

Risky pattern:

```text
Every retry blindly inserts the same rows again -> duplicate data
```

Safer patterns:

```text
Use a unique key
Use upsert / merge
Delete and replace only a clearly defined snapshot
Write to a temporary table, then swap
```

In this demo, the load tasks delete existing rows for the same source timestamp before inserting again. That makes retries safer for the same API snapshot.

In a production historical pipeline, choose stable keys such as:

```text
source + area/station + timestamp
```

Interview-style answer:

```text
Airflow can retry and rerun failed tasks, but the task logic must be idempotent.
Otherwise, retries can create duplicate data or inconsistent outputs.
```


## 19. Try a Real API Failure and Read the Logs

Now we will create a more realistic failure.

Instead of manually writing `raise ValueError(...)`, we will make the taxi task call an API URL that does not exist. This is simple, direct, and close to a real production mistake: a provider URL changed, a path was typed incorrectly, or a configuration value pointed to the wrong endpoint.

In this experiment, the API should return `404 Not Found`. The Airflow task calls `response.raise_for_status()`, so the task should fail and write the HTTP error into the task logs.

### Step 1: start the Day 2.1 mock API

Open a new terminal from the course root folder.

Windows PowerShell:

```powershell
.\.venv\Scripts\Activate.ps1
cd day_2\day_2_1_API
python -m uvicorn mock_serving_api:app --reload --host 127.0.0.1 --port 8011
```

macOS / Linux:

```bash
source ./.venv/bin/activate
cd day_2/day_2_1_API
python -m uvicorn mock_serving_api:app --reload --host 127.0.0.1 --port 8011
```

Open this page to check that the API is running:

```text
http://127.0.0.1:8011/docs
```

Leave this terminal running. It is the local API server that Airflow will call in the failure experiment.

### Step 2: directly visit the wrong URL

First, confirm the failure in the browser before involving Airflow.

Open this wrong URL:

```text
http://127.0.0.1:8011/api/v1/transport/taxi-availability-wrong
```

You should see a `404 Not Found` response. That is the exact failure we want Airflow to capture.

The correct local mock API path would be:

```text
http://127.0.0.1:8011/api/v1/transport/taxi-availability
```

But for this experiment, keep using the wrong path.

### Step 3: point the taxi task to the wrong URL

Before changing anything, find the code that controls the taxi API request.

Open this file:

```text
airflow_project/jobs/config.py
```

Find this setting:

```python
TAXI_API_URL = os.getenv(
    "TAXI_API_URL",
    "https://api.data.gov.sg/v1/transport/taxi-availability",
)
```

This is the normal source: Airflow fetches taxi data from data.gov.sg.

Now open this file:

```text
airflow_project/jobs/taxi_job.py
```

Find the import near the top:

```python
from jobs.config import (
    CLEAN_DIR,
    CURATED_DIR,
    DB_PATH,
    RAW_DIR,
    TAXI_API_URL,
    ensure_dirs,
)
```

For this failure experiment, paste this line after the import block:

```python
TAXI_API_URL = "http://host.docker.internal:8011/api/v1/transport/taxi-availability-wrong"
```

This temporarily overrides the real taxi API URL only for this file.

Why not `127.0.0.1` here?

Airflow is running inside Docker. From inside the Airflow container, `127.0.0.1` means the container itself. `host.docker.internal` points back to your laptop, where the mock API is running.

Now find the API request:

```python
response = requests.get(TAXI_API_URL, timeout=10)
response.raise_for_status()
```

The request goes to the wrong URL. The `raise_for_status()` line turns the `404 Not Found` response into a Python exception, which Airflow records as a task failure.

Save `taxi_job.py`. For this experiment, try triggering the DAG directly from the Web UI. Airflow usually reads the updated Python file when the task starts. If the old behavior still appears, wait a moment and trigger a new DAG run.

### Step 4: trigger the DAG and inspect the failure

In the Airflow UI:

1. Trigger `dashboard_data_pipeline`.
2. Open the DAG run.
3. Watch `fetch_taxi_api`.
4. Open the task logs.
5. Look for `404 Client Error` or `Not Found`.

Because the DAG has retries, the task may first move into an `up_for_retry` state before it finally becomes `failed`.

The weather branch may still succeed. That is useful: Airflow shows exactly which part of the pipeline failed.

This is the point of orchestration. Airflow does not just run Python files; it records task state, retries failed work, keeps logs, and shows where the pipeline stopped.

### Step 5: recover and rerun

Open `airflow_project/jobs/taxi_job.py` and remove or comment out the temporary override:

```python
# TAXI_API_URL = "http://host.docker.internal:8011/api/v1/transport/taxi-availability-wrong"
```

Trigger the DAG again. This time the taxi branch should recover and use the normal data.gov.sg source from `config.py`.

Key idea:

An API failure is not just a Python exception. In a scheduled pipeline, it becomes an operational event: task state, retries, logs, partial success, recovery, and rerun behavior.


## 20. Scheduling

The dashboard pipeline DAG already includes a schedule:

```python
dashboard_data_pipeline: every 5 minutes
```

For learning, it is still useful to trigger the DAG manually first. Manual runs let you see one run clearly.

Common schedule examples:

| Schedule | Meaning |
|---|---|
| `timedelta(minutes=5)` | Every 5 minutes |
| `@hourly` | Every hour |
| `@daily` | Every day |
| `None` | Manual only |


## 21. Common Problems

| Problem | What to try |
|---|---|
| Docker command not found | Start Docker Desktop, then reopen terminal. |
| Airflow UI does not open | Wait a few minutes, then run `docker compose ps`. |
| Port 8080 is busy | Stop the other service using port 8080. |
| DAG is not visible | Check `airflow_project/dags/` file names and scheduler logs. |
| Task is queued forever | Check whether `data_ingestion_pool` exists. |
| API request fails | Check task logs; retry may fix temporary network issues. |
| DuckDB file missing | Trigger the DAG and wait for success. |
| Serving API cannot find DB | Check that `data/basic_ingestion.duckdb` exists in `airflow_project`. |
| Code change does not appear | Restart scheduler, dag processor, and worker. For `.env` or Docker changes, run `docker compose down` and `docker compose up -d`. |

Useful short commands:

```bash
docker compose ps
docker compose logs airflow-scheduler
```


## 22. Stop Airflow

Stop containers:

```bash
docker compose down
```

Full reset, including Airflow metadata database:

```bash
docker compose down --volumes --remove-orphans
```

Remove generated data files if you want to start fresh.

Windows PowerShell:

```powershell
Remove-Item -Recurse -Force .\data\raw, .\data\clean, .\data\curated, .\data\basic_ingestion.duckdb
```

macOS Terminal:

```bash
rm -rf data/raw data/clean data/curated data/basic_ingestion.duckdb
```


## 23. Summary: Interview-Style Review

You have now run a Day 1-style ingestion workflow through Airflow.

A useful way to review Airflow is to connect each concept to a common interview question.

### Question 1: In Airflow, what are DAG, Operator, Task, and Task Instance?

A DAG defines the workflow and dependencies.

An Operator is a template for work, such as `PythonOperator`.

A task is one step in the DAG, such as `fetch_taxi_api` or `load_taxi_counts_to_duckdb`.

A Task Instance is one actual run of that task for one DAG run.

In this lab, the DAG made the pipeline visible:

```text
fetch -> clean -> join -> load
```

### Question 2: What problem does Airflow solve?

Airflow is an orchestration tool.

It does not replace Python, SQL, APIs, or DuckDB. It coordinates when each step should run, what depends on what, what succeeded, what failed, and what should be retried.

In this lab, Airflow helped us manage:

- task dependencies
- schedules
- retries
- logs
- failed tasks
- reruns
- shared-resource control with pools

### Question 3: If an API task fails, what should you check?

Start from the failed task in the Airflow UI.

Check:

- task status
- task logs
- API URL
- timeout error
- HTTP status code
- retry behavior
- whether upstream data is delayed or unavailable

In this lab, the teaching API timeout showed that an API failure becomes an operational event in Airflow: retry, `up_for_retry`, failed task, logs, and recovery.

### Question 4: What is idempotency, and why does it matter?

Idempotency means rerunning a task produces the same final result.

This matters because Airflow may rerun tasks after failure, manual retry, or backfill. If the task is not idempotent, retries can create duplicate rows or inconsistent outputs.

In this lab, the load tasks delete or replace the same timestamp before inserting again. That makes reruns safer.

### Question 5: What is backfill?

Backfill means running a pipeline for past scheduled dates or missing historical periods.

Backfill is useful when:

- the pipeline was turned off
- an upstream API was down
- a bug was fixed and old data must be recomputed
- historical partitions need to be rebuilt

To support backfill, tasks should be parameterized by time, write to clear partitions or timestamps, and be idempotent.

### Final takeaway

A production data pipeline is not just code that runs once.

It needs orchestration: clear DAG structure, task-level responsibility, scheduling, logging, retry behavior, safe reruns, backfill support, and a plan for what to do when upstream systems fail.

---
